# fetch

> Download YouTube xscripts and metadata locally via yt-dlp

In [ ]:
#| default_exp fetch

## Storage design

**Storage location:** `~/.cache/yttoc/{video_id}/` (XDG Base Directory spec — re-downloadable data belongs in cache). Overridable via `--root`.

```
~/.cache/yttoc/
  {video_id}/
    meta.json
    captions.ja.srt        # if Japanese captions exist
    captions.en.srt        # otherwise fall back to English
```

**meta.json fields:**
```json
{
  "id": "kwSVtQ7dziU",
  "title": "Skill Issue: Andrej Karpathy on Code Agents...",
  "channel": "No Priors",
  "duration": 3991,
  "upload_date": "20260320",
  "webpage_url": "https://www.youtube.com/watch?v=kwSVtQ7dziU",
  "description": "Video description with possible manual ToC...",
  "captions": {"ja": "manual"},
  "last_used_at": "2026-04-06T14:22:31+00:00"
}
```

`description` is preserved for use as background context in ToC/summary generation. Many videos include a manual ToC with timestamps in their description.

**Caption selection logic:** try Japanese first (manual `ja`/`ja-*` → auto `ja`/`ja-*`), then English (manual `en`/`en-*` → auto `en`/`en-*`). Error if neither exists.

**CLI:** `yttoc-fetch <url>` fetches one video, prints video_id to stdout. Batch via shell: `cat urls.txt | xargs -I{} yttoc-fetch {}`

## Modularize

In [ ]:
#| export
import json
from datetime import datetime, timezone
from pathlib import Path
from pydantic import BaseModel, Field
import yt_dlp
from yttoc.core import Meta
from yttoc.cache import resolve_root, glob_srt, touch_meta


In [ ]:
#| export
class _YtDlpInfo(BaseModel):
    "Subset of yt-dlp info fields consumed by yttoc.fetch."
    id: str = Field(description="YouTube video ID")
    title: str = Field(description="Video title")
    channel: str = Field(description="Channel name")
    duration: int = Field(ge=0, description="Duration in seconds")
    upload_date: str = Field(description="Upload date in YYYYMMDD format")
    webpage_url: str = Field(description="Canonical YouTube URL")
    description: str = Field(default='', description="Video description (may be empty)")
    language: str | None = Field(default=None, description="Original spoken language code")
    subtitles: dict[str, list[dict[str, object]]] = Field(default_factory=dict,
        description="Manual subtitle tracks by language")
    automatic_captions: dict[str, list[dict[str, object]]] = Field(default_factory=dict,
        description="Automatic caption tracks by language")

def _coerce_video_info(info: dict | _YtDlpInfo) -> _YtDlpInfo:
    "Validate the yt-dlp fields yttoc.fetch depends on."
    if isinstance(info, _YtDlpInfo):
        return info
    return _YtDlpInfo.model_validate(info)

def _pick_lang(tracks: dict,
               base_lang: str = 'en' # Preferred base language (may be a regional tag like 'en-US')
              ) -> str | None: # Best-matching language key actually present in tracks
    "Match base_lang exactly, else its parent family, else a sibling variant."
    tracks = tracks or {}
    if base_lang in tracks: return base_lang
    family = base_lang.split('-', 1)[0]
    if family != base_lang and family in tracks: return family
    for lang in sorted(k for k in tracks if k != 'live_chat'):
        if lang.startswith(f'{family}-'):
            return lang
    return None

def _build_meta(info: dict | _YtDlpInfo, # yt-dlp info dict or validated subset
                lang: str = 'en', # Language that was fetched
                caption_type: str = 'auto' # 'manual' or 'auto'
               ) -> Meta: # Parsed Meta instance
    "Extract fields for meta.json from yt-dlp info."
    info = _coerce_video_info(info)
    meta = Meta(
        id=info.id,
        title=info.title,
        channel=info.channel,
        duration=info.duration,
        upload_date=info.upload_date,
        webpage_url=info.webpage_url,
        description=info.description,
        captions={lang: caption_type},
        last_used_at=datetime.now(timezone.utc),
    )
    return meta

def _download_srt(url: str, info: dict | _YtDlpInfo, out_dir: Path
                 ) -> tuple[Path, str, str]: # (srt_path, selected_lang, caption_type)
    "Download SRT captions in the video's original spoken language."
    info = _coerce_video_info(info)
    base_lang = info.language
    if not base_lang:
        raise ValueError(f"Cannot determine original language for {info.id}")
    manual_lang = _pick_lang(info.subtitles, base_lang)
    auto_lang = _pick_lang(info.automatic_captions, base_lang)
    selected_lang = manual_lang or auto_lang
    if selected_lang is None:
        raise ValueError(f"No {base_lang} captions available for {info.id}")

    sub_opt = 'writesubtitles' if manual_lang else 'writeautomaticsub'
    opts = {
        'skip_download': True, 'quiet': True,
        sub_opt: True,
        'subtitleslangs': [selected_lang],
        'subtitlesformat': 'srt',
        'outtmpl': str(out_dir / f'captions_{selected_lang}.%(ext)s'),
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([url])

    srt_path = out_dir / f'captions.{selected_lang}.srt'
    matches = glob_srt(out_dir, f'captions_{selected_lang}*.srt')
    if not matches:
        raise FileNotFoundError(f"yt-dlp did not write an srt caption for {info.id}")
    if len(matches) > 1:
        raise ValueError(f"Ambiguous caption files: {', '.join(p.name for p in matches)}")
    if matches[0] != srt_path:
        matches[0].replace(srt_path)
    caption_type = 'manual' if manual_lang else 'auto'
    return srt_path, selected_lang, caption_type


In [ ]:
# Test: helpers
from pydantic import ValidationError

assert _pick_lang({'en': []}) == 'en'
assert _pick_lang({'live_chat': [], 'en-US': []}) == 'en-US'
assert _pick_lang({'ja': []}) is None
assert _pick_lang({'ja': []}, 'ja') == 'ja'
assert _pick_lang({'ja-JP': []}, 'ja') == 'ja-JP'
assert _pick_lang({'en': []}, 'ja') is None
# Regional tag → parent family, then sibling variants
assert _pick_lang({'en': []}, 'en-US') == 'en'
assert _pick_lang({'en-GB': []}, 'en-US') == 'en-GB'
assert _pick_lang({'en': [], 'en-US': []}, 'en-US') == 'en-US'

# Test: _build_meta
fake_info = {'id': 'X', 'title': 't', 'channel': 'c', 'duration': 1,
             'upload_date': '20260101', 'webpage_url': 'u',
             'description': 'desc', 'subtitles': {}, 'automatic_captions': {'en': []}}
typed = _coerce_video_info({
    'id': 'Y', 'title': 'typed', 'channel': 'chan', 'duration': 2,
    'upload_date': '20260101', 'webpage_url': 'https://y.com/Y'})
assert typed.description == ''
assert typed.language is None
assert typed.subtitles == {}
assert typed.automatic_captions == {}
try:
    _coerce_video_info({'id': 'BAD', 'title': 't', 'duration': 1,
                        'upload_date': '20260101', 'webpage_url': 'u'})
except ValidationError:
    pass
else:
    assert False, 'expected ValidationError for missing channel in yt-dlp info subset'

m = _build_meta(fake_info, lang='en', caption_type='auto')
assert m.captions == {'en': 'auto'}
assert m.description == 'desc'
m_typed = _build_meta(typed, lang='en', caption_type='auto')
assert m_typed.id == 'Y'

m_ja = _build_meta(fake_info, lang='ja', caption_type='manual')
assert m_ja.captions == {'ja': 'manual'}
print('ok')

In [ ]:
# Test: _download_srt language resolution (unit-level, no network)
# Picks the actually-matched caption key from info['language'], preferring manual
# over auto. For regional tags (e.g. 'en-US') falls back to the parent family.

def _resolve_lang(info):
    "Simulate _download_srt's language resolution without downloading."
    base = info.get('language')
    if not base:
        return None, None
    manual = _pick_lang(info.get('subtitles', {}), base)
    auto = _pick_lang(info.get('automatic_captions', {}), base)
    sel = manual or auto
    if sel is None:
        return None, None
    return sel, ('manual' if manual else 'auto')

# English video with manual en subs → en manual
assert _resolve_lang({'language': 'en', 'subtitles': {'en': []}, 'automatic_captions': {}}) == ('en', 'manual')

# English video with only auto en → en auto
assert _resolve_lang({'language': 'en', 'subtitles': {}, 'automatic_captions': {'en': [], 'ja': []}}) == ('en', 'auto')

# Japanese video → picks ja, ignores en
assert _resolve_lang({'language': 'ja', 'subtitles': {}, 'automatic_captions': {'ja': [], 'en': []}}) == ('ja', 'auto')

# Original ja with ja-JP variant — records the matched key
assert _resolve_lang({'language': 'ja', 'subtitles': {'ja-JP': []}, 'automatic_captions': {}}) == ('ja-JP', 'manual')

# Regional base language falls back to parent family — records 'en', not 'en-US'
assert _resolve_lang({'language': 'en-US', 'subtitles': {}, 'automatic_captions': {'en': []}}) == ('en', 'auto')

# English video but no en captions at all → None (no cross-language fallback)
assert _resolve_lang({'language': 'en', 'subtitles': {}, 'automatic_captions': {'ja': []}}) == (None, None)

# Missing language field → None
assert _resolve_lang({'subtitles': {'en': []}, 'automatic_captions': {}}) == (None, None)

print('ok')

In [ ]:
#| export
def get_video_info(url: str # YouTube video URL
                  ) -> dict: # yt-dlp info dict
    "Extract video metadata and caption info without downloading."
    with yt_dlp.YoutubeDL({'skip_download': True, 'quiet': True}) as ydl:
        return ydl.extract_info(url, download=False)

In [ ]:
#| network
# Test: get_video_info smoke test (requires network)
info = get_video_info('https://www.youtube.com/watch?v=kwSVtQ7dziU')
for k in ['id', 'title', 'channel', 'duration', 'upload_date', 'webpage_url']:
    assert k in info, f'missing {k}'
assert info['id'] == 'kwSVtQ7dziU'
print('ok')

In [ ]:
#| export
def fetch_video(url: str, # YouTube video URL
                info: dict | _YtDlpInfo, # Result of get_video_info or validated subset
                root: str | Path = None, # Root download directory (default: ~/.cache/yttoc)
               ) -> Path: # Path to video directory
    "Save metadata and srt captions for one video in its original spoken language."
    info = _coerce_video_info(info)
    root = resolve_root(root)
    out_dir = root / info.id
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_path = out_dir / 'meta.json'
    if glob_srt(out_dir) and meta_path.exists():
        touch_meta(info.id, root)
        return out_dir

    _srt_path, lang, caption_type = _download_srt(url, info, out_dir)
    meta = _build_meta(info, lang=lang, caption_type=caption_type)
    meta_path.write_text(meta.model_dump_json(indent=2), encoding='utf-8')
    return out_dir


In [ ]:
#| eval: false
# Test: fetch_video (requires network)
import shutil
test_root = '/tmp/yttoc_test'
shutil.rmtree(test_root, ignore_errors=True)

url = 'https://www.youtube.com/watch?v=kwSVtQ7dziU'
info = get_video_info(url)
out = fetch_video(url, info, root=test_root)
assert (out / 'meta.json').exists()
assert glob_srt(out)

meta = json.loads((out / 'meta.json').read_text(encoding='utf-8'))
assert meta['id'] == 'kwSVtQ7dziU'
assert 'last_used_at' in meta
assert 'captions' in meta

shutil.rmtree(test_root)
print('ok')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse

@call_parse
def yttoc_fetch(url: str, # YouTube video URL
                root: str = None, # Root download directory (default: ~/.cache/yttoc)
               ):
    "Fetch metadata and captions for a single YouTube video in its original spoken language."
    info = get_video_info(url)
    out = fetch_video(url, info, root)
    print(info['id'])

In [ ]:
#| export
from yttoc.core import fmt_duration as _fmt_duration

In [ ]:
# Test: _fmt_duration (imported from core)
assert _fmt_duration(3991) == '1:06:31'
assert _fmt_duration(195) == '3:15'
assert _fmt_duration(0) == '0:00'
print('ok')

In [ ]:
#| export
def _render_list(items: list[tuple['Meta', str]] # Sorted list of (meta, langs_string)
                ) -> str: # Formatted listing, one line per video
    "Render the yttoc_list table from pre-sorted items."
    if not items:
        return ''
    lines = []
    for meta, langs in items:
        ts = meta.last_used_at.isoformat()[:16].replace('T', ' ')
        dur = _fmt_duration(meta.duration)
        lines.append(f"{meta.id}  {ts}  {dur:>8}  [{langs}]  {meta.title}")
    return '\n'.join(lines)

@call_parse
def yttoc_list(root: str = None, # Root directory (default: ~/.cache/yttoc)
              ):
    "List cached videos sorted by last used."
    root = resolve_root(root)
    if not root.exists(): return

    items = []  # list of (meta: Meta, langs: str)
    for d in root.iterdir():
        if not d.is_dir(): continue
        meta_path = d / 'meta.json'
        srt_files = glob_srt(d)
        if not (meta_path.exists() and srt_files): continue
        meta = Meta.model_validate_json(meta_path.read_text(encoding='utf-8'))
        captions = meta.captions or {p.stem.split('.', 1)[1]: '?' for p in srt_files}
        langs = ','.join(sorted(captions.keys()))
        items.append((meta, langs))

    items.sort(key=lambda x: x[0].last_used_at, reverse=True)
    out = _render_list(items)
    if out:
        print(out)


In [ ]:
# Test: yttoc_list cache-walking (ordering, filtering, lang recovery)
import io, contextlib
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    root = Path(d)
    # Video A: older, English only
    a = root / 'AAAA'
    a.mkdir()
    (a / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (a / 'meta.json').write_text(json.dumps({
        'id': 'AAAA', 'title': 'Old video', 'channel': 'Ch', 'duration': 195,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=AAAA',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2026-01-01T00:00:00+00:00',
    }))
    # Video B: newer, Japanese
    b = root / 'BBBB'
    b.mkdir()
    (b / 'captions.ja.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nこんにちは\n')
    (b / 'meta.json').write_text(json.dumps({
        'id': 'BBBB', 'title': 'New video', 'channel': 'Ch', 'duration': 3991,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=BBBB',
        'description': '', 'captions': {'ja': 'manual'},
        'last_used_at': '2026-04-06T15:00:00+00:00',
    }))
    # Video C: incomplete (no srt) — should be skipped
    c = root / 'CCCC'
    c.mkdir()
    (c / 'meta.json').write_text('{"id":"CCCC"}')

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_list(root=str(root))
    out = buf.getvalue()
    lines = [l for l in out.strip().splitlines() if l.strip()]

    # Cache-walking: incomplete dirs skipped
    assert len(lines) == 2, f'expected 2 lines, got {len(lines)}'
    assert 'CCCC' not in out
    # Sorting: most recently used first
    assert out.index('BBBB') < out.index('AAAA')

# Legacy: lang recovery from srt filename when captions dict is empty
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'LEGACY'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'LEGACY', 'title': 'Legacy video', 'channel': 'Ch', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=LEGACY',
        'description': '', 'captions': {},
        'last_used_at': '2026-01-01T00:00:00+00:00',
    }))
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_list(root=str(root))
    out = buf.getvalue()
    assert 'LEGACY' in out
    assert '[en]' in out  # detected from filename

print('ok')


In [ ]:
# Test: yttoc_list rejects a corrupted meta.json (invalid caption type)
import io, contextlib
from tempfile import TemporaryDirectory
from pydantic import ValidationError

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'BAD1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'BAD1', 'title': 'T', 'channel': 'C', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://y.com',
        'description': '', 'captions': {'en': 'autop'},
        'last_used_at': '2026-01-01T00:00:00+00:00'}))

    try:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            yttoc_list(root=str(root))
    except ValidationError:
        pass
    else:
        assert False, 'expected ValidationError for invalid caption type in meta.json'
print('ok')


In [ ]:
# Test: _render_list returns formatted lines with no stdout capture
from yttoc.fetch import _render_list
from yttoc.core import Meta
from datetime import datetime, timezone

meta_a = Meta(id='AAAA', title='First Video', channel='C', duration=3661,
              upload_date='20260101', webpage_url='https://youtube.com/watch?v=AAAA',
              description='', captions={'en': 'auto'},
              last_used_at=datetime(2026,4,20,12,0,tzinfo=timezone.utc))
meta_b = Meta(id='BBBB', title='Second Video', channel='C', duration=120,
              upload_date='20260101', webpage_url='https://youtube.com/watch?v=BBBB',
              description='', captions={'en': 'auto', 'ja': 'manual'},
              last_used_at=datetime(2026,4,19,8,0,tzinfo=timezone.utc))

items = [(meta_a, 'en'), (meta_b, 'en,ja')]
out = _render_list(items)
lines = out.strip().split('\n')
assert len(lines) == 2
assert 'AAAA' in lines[0]
assert '1:01:01' in lines[0]
assert 'First Video' in lines[0]
assert '[en]' in lines[0]
assert 'BBBB' in lines[1]
assert '2:00' in lines[1]
assert '[en,ja]' in lines[1]

# Empty list returns empty string
assert _render_list([]) == ''
print('ok')


In [ ]:
# Test: fetch_video cache hit with existing captions
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    fake_info = {'id': 'TEST123', 'title': 't', 'channel': 'c',
                 'duration': 60, 'upload_date': '20260101',
                 'webpage_url': 'https://example.com', 'description': '',
                 'subtitles': {}, 'automatic_captions': {'en': [{'ext': 'srt'}]}}
    out_dir = Path(d) / 'TEST123'
    out_dir.mkdir()
    (out_dir / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\ntest\n')
    (out_dir / 'meta.json').write_text(json.dumps({
        'id': 'TEST123', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://example.com',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    result = fetch_video('https://example.com', fake_info, root=d)
    assert result.name == 'TEST123'
    meta = json.loads((result / 'meta.json').read_text())
    assert 'last_used_at' in meta

# Cache hit also works with ja srt
with TemporaryDirectory() as d:
    out_dir = Path(d) / 'TEST456'
    out_dir.mkdir()
    (out_dir / 'captions.ja.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nテスト\n')
    (out_dir / 'meta.json').write_text(json.dumps({
        'id': 'TEST456', 'title': 't', 'channel': 'c', 'duration': 60,
        'upload_date': '20260101', 'webpage_url': 'https://example.com',
        'description': '', 'captions': {'ja': 'manual'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    result = fetch_video('https://example.com',
        {'id': 'TEST456', 'title': 't', 'channel': 'c', 'duration': 60,
         'upload_date': '20260101', 'webpage_url': 'u', 'description': '',
         'subtitles': {'ja': []}, 'automatic_captions': {}}, root=d)
    meta = json.loads((result / 'meta.json').read_text())
    assert meta['captions'] == {'ja': 'manual'}
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()